# Webscraping a ABC

ABC parece scrapeable con `request` y `BeautifulSoup`. La página de categoría (nacional, internacional o cultura) contiene artículos en HTML estático. El patrón más útil parece ser:

```HTML
<article>
  ...
  <a class="v-a-lnk" href="..." title="...">
  ...
  <h2 class="v-a-t">
  ...
  <p class="v-a-s-t">...</p>
</article>
```

| Campo     | De dónde saldría                           |
| --------- | ------------------------------------------ |
| Link      | `a.v-a-lnk["href"]` o `h2.v-a-t a["href"]` |
| Título    | atributo `title` o texto del `h2`          |
| Subtítulo | `p.v-a-s-t`                                |
| Categoría | la asignamos nosotros: `"Nacional"`        |
| Periódico | fijo: `"ABC"`                              |

Dentro de cada artículo, ABC nos da casi todo ya limpio dentro de:

```html
<script type="application/ld+json" id="evo-swg-markup">
```

De aquí podemos extraer directamente:

| Campo JSON nuestro | Campo en ABC                                  |
| ------------------ | --------------------------------------------- |
| `Link`             | `mainEntityOfPage["@id"]`                     |
| `Periódico`        | fijo: `"ABC"`                                 |
| `Fecha`            | `datePublished`, recortando solo `YYYY-MM-DD` |
| `Título`           | `headline`                                    |
| `Subtítulo`        | `description`                                 |
| `Categoría`        | la asignamos según la sección: `"Nacional"`   |
| `Contenido`        | `articleBody`                                 |

Se consultó el archivo robots.txt del medio. Las secciones públicas usadas para el estudio no aparecen bloqueadas para User-agent general. El scraping se limita a un máximo de 15 artículos diarios, con pausas entre peticiones, sin acceder a zonas privadas, de pago ni endpoints bloqueados. Los datos se emplean exclusivamente con fines académicos.


In [20]:
# Cargamos las librerías necesarias

import requests               #para descargar HTML
from bs4 import BeautifulSoup #para parsear el HTML
import json                   #para guardar los datos en formato JSON
import re                     #limpiar texto con expresiones regulares
import time                   #para hacer pausas entre peticiones
from datetime import datetime #para manejar fechas
import pandas as pd           #para revisar resultados en formato tabular
import os                     #para manejar archivos y directorios

In [21]:
# Configuaramos headers para simular un navegador y que ABC no piense que somos un bot
headers = {
    'User-Agent': 'ProyectoAcademico/1.0'
}

In [ ]:
#probamos la conexión a la página apartado de noticias nacionales
#url = 'https://www.abc.es/espana/'

#r=requests.get(url, headers=headers)

#print(r.status_code) #debería imprimir 200 si todo va bien
#print(r.text[:500]) #imprime los primeros 500 caracteres del HTML para verificar que se ha descargado correctamente

200
<!DOCTYPE html>
<html xmlns="http://www.w3.org/1999/xhtml" xmlns:og="http://ogp.me/ns#" xmlns:fb="http://www.facebook.com/2008/fbml" lang="es">
<head>
<meta charset="utf-8"><meta name="lang" content="es" />
<meta http-equiv="X-UA-Compatible" content="IE=edge,chrome=1">
<meta name="viewport" content="width=device-width, initial-scale=1.0, user-scalable=no">  
  <link rel="preconnect" href="https://static.vocstatic.com" />
  <link rel="preconnect" href="https://static1.vocstatic.com" />
  <link re


In [23]:
#Funcion para descargar una página y parsearla con BeautifulSoup
def obtener_soup(url):
    response = requests.get(url, headers=headers, timeout=10) #timeout para evitar que se quede colgado si la página no responde
    response.raise_for_status() #lanzar excepción si la respuesta no es 200
    return BeautifulSoup(response.text, 'html.parser')

In [ ]:
#Funcion para obtener links de una categoría dada su URL
def obtener_links_categoria(url_categoria, max_links=5):
    soup = obtener_soup(url_categoria) #llamamos a la función para obtener el soup de la página de la categoría
    links = [] #lista para almacenar los links de los artículos

    for a in soup.select("a.v-a-lnk"):
        href = a.get('href') #obtenemos el atributo href del enlace

        if not href:
            continue #si no hay href, saltamos este enlace

        if href.startswith(url_categoria):
            url_valida = href #si el link ya es absoluto, lo usamos tal cual
        elif href.startswith('/'):
            url_valida = url_categoria + href #si el link es relativo, lo convertimos a absoluto
        else:
            continue #si el link no es ni absoluto ni relativo, lo ignoramos

        if url_valida not in links and url_valida.endswith(".html"):
            links.append(url_valida) #agregamos el link a la lista si no está repetido y termina en .html
        
        if len(links) >= max_links:
            break #si ya tenemos suficientes links, salimos del bucle

    return links

#Probamos la función con la categoría de noticias nacionales
#url_categoria = 'https://www.abc.es/espana/'

#links_obtenidos = obtener_links_categoria(url_categoria, max_links=5)
#print("Links obtenidos:")
#for link in links_obtenidos:
#    print(link)
    
    

Links obtenidos:
https://www.abc.es/espana/andalucia/cordoba/cordoba-cf/cordoba-clasificacion-tras-20260426104233-nts.html
https://www.abc.es/espana/andalucia/cordoba/cordoba-cf/notas-jugadores-cordoba-tras-triunfo-ante-sporting-20260426170324-nts.html
https://www.abc.es/espana/madrid/retrasos-trenes-alta-velocidad-atocha-incidencia-tecnica-20260426225235-nt.html
https://www.abc.es/espana/castilla-la-mancha/toledo/virgen-cabeza-reencuentra-tajo-toledano-puente-san-20260426212908-nt.html
https://www.abc.es/espana/castilla-la-mancha/toledo/deportes/toledano-david-magan-octavo-media-maraton-madrid-20260426220815-nt.html


In [ ]:
#Función para extraer un artículo dado su URL

def extraer_articulo(url_articulo, categoria):
    soup = obtener_soup(url_articulo) #obtenemos el soup de la página del artículo

    script = soup.find("script", id="evo-swg-markup") #buscamos el script que contiene los datos estructurados del artículo

    if script is None:
        print(f"No se encontró el script con datos estructurados en {url_articulo}")
        return None
    
    data = json.loads(script.string) #cargamos el contenido del script como JSON

    fecha_completa = data.get('datePublished') #obtenemos la fecha de publicación completa

    fecha = fecha_completa[:10] if fecha_completa else None #extraemos solo la parte de la fecha (YYYY-MM-DD)

    articulo = {
        "Link": url_articulo, #obtenemos el link del artículo
        "Periódico": "ABC", #el periódico es ABC
        "Fecha": fecha, #la fecha de publicación del artículo
        "Título": data.get('headline'), #el título del artículo
        "Subtítulo": data.get('description'), #el subtítulo del artículo
        "Categoria": categoria, #la categoría a la que pertenece el artículo
        "Contenido": data.get('articleBody') #el contenido del artículo
    }

    return articulo

#probar con el primer artículo obtenido

#articulo_ejemplo = extraer_articulo(links_obtenidos[0], "Nacional")

#print("Artículo extraído:")
#print(json.dumps(articulo_ejemplo, indent=2, ensure_ascii=False))

Artículo extraído:
{
  "Link": "https://www.abc.es/espana/andalucia/cordoba/cordoba-cf/cordoba-clasificacion-tras-20260426104233-nts.html",
  "Periódico": "ABC",
  "Fecha": "2026-04-26",
  "Título": "Así está el Córdoba CF en la clasificación tras el triunfo ante el Sporting",
  "Subtítulo": "El conjunto blanquiverde logra la permanencia matemática y se acerca a los puestos de play off",
  "Categoria": "Nacional",
  "Contenido": "El Córdoba CF ha disputado este domingo la trigésimo séptima jornada  de la Liga Hypermotion ante el Sporting de Gijón en El Arcángel. El conjunto blanquiverde realizó un partido muy serio en el que supo sobreponerse con claridad ante el conjunto asturiano (3- ... 2). Pese al tanto inicial de Otero, el cuadro cordobesista logró reponerse con las dianas de Kevin Medina y Jacobo. Albarrán culminó la fiesta pese al tanto de Gelabert y seguir soñando por cotas más altas Tras este resultado, los de Iván Ania se verán las caras el próximo  sábado 2 de mayo a las 18.

In [ ]:
#Scrapear 5 noticias nacionales

#articulos_nacionales = []

#for link in links_obtenidos:
#    articulo = extraer_articulo(link, "Nacional")
#    if articulo:
#        articulos_nacionales.append(articulo)
#    time.sleep(1) #pausa de 1 segundo entre peticiones para no saturar el servidor

#df_nacionales = pd.DataFrame(articulos_nacionales)
#df_nacionales

,Link,Periódico,Fecha,Título,Subtítulo,Categoria,Contenido
0,https://www.abc.es/espana/andalucia/cordoba/co...,ABC,2026-04-26,Así está el Córdoba CF en la clasificación tra...,El conjunto blanquiverde logra la permanencia ...,Nacional,El Córdoba CF ha disputado este domingo la tri...
1,https://www.abc.es/espana/andalucia/cordoba/co...,ABC,2026-04-26,Las notas de los jugadores del Córdoba CF tras...,Los pupilos de Iván Ania firmaron una actuació...,Nacional,La vida sigue siendo de color de rosa para el ...
2,https://www.abc.es/espana/madrid/retrasos-tren...,ABC,2026-04-26,Retrasos de los trenes de alta velocidad en At...,Está afectada la salida y la llegada de convoy...,Nacional,Una nueva incidencia en la infraestructura de ...
3,https://www.abc.es/espana/castilla-la-mancha/t...,ABC,2026-04-26,La Virgen de la Cabeza se reencuentra con el T...,A la siete de la tarde la Virgen salió de su e...,Nacional,La Virgen de la Cabeza ha vuelto a cruzar por ...
4,https://www.abc.es/espana/castilla-la-mancha/t...,ABC,2026-04-26,"El toledano David Magán, octavo en la Media Ma...",El atleta del Running San Miguel firmó 1:10:25...,Nacional,"El toledano David Magán, del club Running San ..."


In [ ]:
# Automatizar las 3 categorías principales: Nacional, Internacional y Deportes

ARCHIVO_JSON = "data/ABC.json"

# Funcion para cargar datos existentes desde el archivo JSON

def cargar_json(nombre_archivo):
    try:
        with open(nombre_archivo, "r", encoding="utf-8") as f: #abrimos el archivo en modo lectura
            return json.load(f)
    except FileNotFoundError: #si el archivo no existe, devolvemos una lista vacía
        return []
    
# Funcion para guardar datos en el archivo JSON
def guardar_json(nombre_archivo, articulos):
    with open(nombre_archivo, "w", encoding="utf-8") as f: #abrimos el archivo en modo escritura
        json.dump(articulos, f, ensure_ascii=False, indent=2) #guardamos la lista de artículos en formato JSON con indentación para que sea legible

# Funcion para añadir evitando duplicados por Link

def actualizar_json(nombre_archivo, nuevos_articulos):
    articulos_existentes = cargar_json(nombre_archivo) #cargamos los artículos existentes desde el archivo JSON

    links_existentes = {articulo["Link"] for articulo in articulos_existentes} #creamos un conjunto de los links existentes para facilitar la comparación

    articulos_a_anadir = [
        articulo for articulo in nuevos_articulos #filtramos los nuevos artículos para quedarnos solo con los que no están ya en el archivo JSON
        if articulo["Link"] not in links_existentes #si el link del artículo no está en el conjunto de links existentes, lo añadimos a la lista de artículos a añadir
    ]

    articulos_totales = articulos_existentes + articulos_a_anadir #combinamos los artículos existentes con los nuevos artículos que no estaban repetidos

    guardar_json(nombre_archivo, articulos_totales) #guardamos la lista total de artículos en el archivo JSON

    print(f"Artículos nuevos añadidos: {len(articulos_a_anadir)}") 
    #print(f"Total artículos en {nombre_archivo}: {len(articulos_totales)}")


In [35]:
categorias = {
    "Nacional": "https://www.abc.es/espana/",
    "Internacional": "https://www.abc.es/internacional/",
    "Cultura": "https://www.abc.es/cultura/"
}

for categoria, url_categoria in categorias.items():
    print(f"Procesando categoría: {categoria}")
    links = obtener_links_categoria(url_categoria, max_links=5) #obtenemos los links de la categoría
    articulos = []
    
    for link in links:
        articulo = extraer_articulo(link, categoria) #extraemos el artículo de cada link
        if articulo:
            articulos.append(articulo)
        time.sleep(1) #pausa de 1 segundo entre peticiones para no saturar el servidor
    
    actualizar_json(ARCHIVO_JSON, articulos) #actualizamos el archivo JSON con los nuevos artículos obtenidos

#comprobamos que se han guardado correctamente los datos en el archivo JSON
articulos_guardados = cargar_json(ARCHIVO_JSON) #cargamos los artículos guardados desde el archivo JSON
print(f"Total artículos guardados en {ARCHIVO_JSON}: {len(articulos_guardados)}") #imprimimos el total de artículos guardados

Procesando categoría: Nacional
Artículos nuevos añadidos: 5
Total artículos en data/ABC.json: 5
Procesando categoría: Internacional
Artículos nuevos añadidos: 5
Total artículos en data/ABC.json: 10
Procesando categoría: Cultura
Artículos nuevos añadidos: 5
Total artículos en data/ABC.json: 15
Total artículos guardados en data/ABC.json: 15
